<a href="https://colab.research.google.com/github/pelinbalci/SLM-FineTune/blob/main/Part_10_GPU_Requirement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook aims to explain the required GPU for finetuning a model. Of course there are some assumptions and it is not the exact calculation but it gives you an idea what type of GPU do you need.

I will use Qwen/Qwen2.5-3B model for all calculations. At the end of the notebook you may find comparison for different models for full fine-tuning, LoRA and QLoRA GPU needs.


*This notebook is prepared with the help of ChatGPT 5.2 and Claude Opus 4.5.*

>[Introduction](#scrollTo=u6qsYQO34NAi)

>[The Memory Components](#scrollTo=JcaaLEiA4uU2)

>>[Step 1: Model Weights](#scrollTo=q2p3gC6wM9Gm)

>>[Step 2: Gradients](#scrollTo=bN80Mf4gN5Bv)

>>[Step 3: Optimizer Step](#scrollTo=gZJ5CmbuOE91)

>>[Step 4: Activations](#scrollTo=aKhEg2FuOfQX)

>>>[Step 4: Details for Normal vs Gradient Checkpoint](#scrollTo=AhaQm2548uVU)

>>[Step 5: CUDA Overhead](#scrollTo=WpWflm85PXUa)

>>[Calcuate Full Finetuning](#scrollTo=065QQsym-Eq_)

>>[Important Note](#scrollTo=XJa2_Ar97uAZ)

>[LoRA Calculation](#scrollTo=J88xbV1cBuC9)

>>[STEP 1: Calculate LoRA Parameters](#scrollTo=JHcAuMU5CEmZ)

>>[STEP 2: Base Model Weights (FROZEN, still in memory)](#scrollTo=c0PQb2isC2Ui)

>>[STEP 3: LoRA Weights (trainable)](#scrollTo=AvXJZ-tyDALe)

>>[STEP 4: Gradients (LoRA ONLY!)](#scrollTo=QSL-8X8XDMsx)

>>[STEP 5: Optimizer States (LoRA ONLY!)](#scrollTo=_w4JHrI5DYsO)

>>[STEP 6: Activations (similar to full fine-tuning)](#scrollTo=RjjNm5qTDwcz)

>>[STEP 7: CUDA Overhead](#scrollTo=N0uD1YPSHjGi)

>>[STEP 8: TOTAL](#scrollTo=lLLmt2PoHsAV)

>[Comparison for Full Finetuning & LoRA](#scrollTo=Q9ACI2ZVIIY1)

>[QLoRA Calculation](#scrollTo=1CfQeSuwKAv3)

>>[STEP 1: Calculate QLoRA Parameters (same as LoRA)](#scrollTo=AapLsK7OKH3w)

>>[STEP 2: Base Model Weights (4-BIT QUANTIZED!) ← KEY DIFFERENCE!](#scrollTo=07MRkZ_SKRwI)

>>[STEP 3: QLoRA Weights (trainable, FP16/BF16 - NOT quantized!)](#scrollTo=VvLgyrcKKZww)

>>[STEP 4: Gradients (LoRA ONLY!)](#scrollTo=n8aqSJX1Kose)

>>[STEP 5: Optimizer States (LoRA ONLY!)](#scrollTo=GColUgqkKrOd)

>>[STEP 6: Activations (slightly higher due to dequantization)](#scrollTo=aDdU52MLKw18)

>>[STEP 7: CUDA Overhead](#scrollTo=zbQ6XEHQLDRz)

>>[STEP 8: TOTAL](#scrollTo=3DxRjvmlLJFd)

>[Comparison for Full Finetuning & LoRA & QLoRA](#scrollTo=SH7ebO_GOwB0)

>[Full Calculation for QLoRA and LoRA](#scrollTo=PWkbUS7dN5p6)

>[Compare FineTuning Methods for a Single Model](#scrollTo=ghTvJREeQ75r)

>[Run Comparison for Different Models](#scrollTo=pvEx77_SUvlA)



# Introduction

**GPU Specifications Reference**

First, let's understand what we're working with:

In [ ]:
# GPU Specifications Database
GPU_SPECS = {
    # GPU Name: (VRAM in GB, Typical Cost/Hour, Notes)

    # Consumer GPUs (for local development)
    'RTX 3060':     {'vram_gb': 12, 'cost_hr': 0.00, 'tier': 'Consumer'},
    'RTX 3080':     {'vram_gb': 10, 'cost_hr': 0.00, 'tier': 'Consumer'},
    'RTX 3090':     {'vram_gb': 24, 'cost_hr': 0.00, 'tier': 'Consumer'},
    'RTX 4090':     {'vram_gb': 24, 'cost_hr': 0.00, 'tier': 'Consumer'},

    # Cloud GPUs (what you use)
    'T4':           {'vram_gb': 16, 'cost_hr': 0.35, 'tier': 'Cloud'},  # Colab
    'L4':           {'vram_gb': 24, 'cost_hr': 0.80, 'tier': 'Cloud'},  # Lightning.ai
    'A10G':         {'vram_gb': 24, 'cost_hr': 1.00, 'tier': 'Cloud'},
    'V100':         {'vram_gb': 16, 'cost_hr': 2.50, 'tier': 'Cloud'},
    'V100-32GB':    {'vram_gb': 32, 'cost_hr': 3.00, 'tier': 'Cloud'},
    'A100-40GB':    {'vram_gb': 40, 'cost_hr': 3.50, 'tier': 'Cloud'},
    'A100-80GB':    {'vram_gb': 80, 'cost_hr': 5.00, 'tier': 'Cloud'},
    'H100':         {'vram_gb': 80, 'cost_hr': 8.00, 'tier': 'Cloud'},
}

**Bytes Per Parameter**

This is the foundation of all memory calculations:

In [ ]:
# How many bytes does each number format use?
BYTES_PER_PARAM = {
    'fp32': 4,      # Full precision (32 bits = 4 bytes)
    'fp16': 2,      # Half precision (16 bits = 2 bytes)
    'bf16': 2,      # Brain float 16 (same size, different range)
    'int8': 1,      # 8-bit quantized
    'int4': 0.5,    # 4-bit quantized (NF4 in QLoRA)
}


Let's use them to calculate the gb of weights in a model.

In [ ]:
def calculate_model_weights_memory(params_billions, precision='fp16'):
    """
    Calculate memory for model weights only.

    Example:
        3B params in FP16 = 3 × 10^9 × 2 bytes = 6 GB
    """
    params = params_billions * 1e9
    bytes_needed = params * BYTES_PER_PARAM[precision]
    gb = bytes_needed / (1024**3)
    return gb

# Test it
print("Model Weights Memory:")
print(f"  3B FP32: {calculate_model_weights_memory(3, 'fp32'):.1f} GB")
print(f"  3B FP16: {calculate_model_weights_memory(3, 'fp16'):.1f} GB")
print(f"  3B INT4: {calculate_model_weights_memory(3, 'int4'):.1f} GB")

Model Weights Memory:
  3B FP32: 11.2 GB
  3B FP16: 5.6 GB
  3B INT4: 1.4 GB


We can use 1024**3 or 1e9 to turn the bytes to gb.

**Find details of a Model**

In [ ]:
from transformers import AutoConfig

# Load config without downloading the full model
config = AutoConfig.from_pretrained("Qwen/Qwen2.5-3B")

print(f"Hidden Dimension: {config.hidden_size}")
print(f"Number of Layers:  {config.num_hidden_layers}")
print(f"Intermediate Size: {config.intermediate_size}")
print(f"Number of Heads:   {config.num_attention_heads}")
print(f"Vocab Size:        {config.vocab_size}")

Hidden Dimension: 2048
Number of Layers:  36
Intermediate Size: 11008
Number of Heads:   16
Vocab Size:        151936


You can see the total model parameters via downloading it:


    from transformers import AutoModelForCausalLM

    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-3B",
        torch_dtype="auto",
        device_map="meta"  # avoids loading weights into GPU/CPU
    )

    total_params = sum(p.numel() for p in model.parameters())

    print(f"Total parameters: {total_params:,}")
    print(f"= {total_params / 1e9:.2f}B")


Or, if the model in hugging face, you can directly call the model_info. It gives you bunch of information about model and one of them is safetensors which gives the total number of parameters in the model! Let's see:

In [ ]:
from huggingface_hub import model_info

info = model_info("Qwen/Qwen2.5-3B")
print(info)


ModelInfo(id='Qwen/Qwen2.5-3B', author='Qwen', sha='3aab1f1954e9cc14eb9509a215f9e5ca08227a9b', created_at=datetime.datetime(2024, 9, 15, 12, 17, 3, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2024, 9, 20, 7, 58, tzinfo=datetime.timezone.utc), private=False, disabled=False, downloads=219730, downloads_all_time=None, gated=False, gguf=None, inference=None, inference_provider_mapping=None, likes=163, library_name=None, tags=['safetensors', 'qwen2', 'text-generation', 'conversational', 'en', 'arxiv:2407.10671', 'license:other', 'region:us'], pipeline_tag='text-generation', mask_token=None, card_data={'base_model': None, 'datasets': None, 'eval_results': None, 'language': ['en'], 'library_name': None, 'license': 'other', 'license_name': 'qwen-research', 'license_link': 'https://huggingface.co/Qwen/Qwen2.5-3B/blob/main/LICENSE', 'metrics': None, 'model_name': None, 'pipeline_tag': 'text-generation', 'tags': None}, widget_data=[{'text': 'Hi, what can you help me with?'}, {'

In [ ]:
print(info.safetensors.total)                 # 3085938688
print(info.safetensors.parameters)            # {'BF16': 3085938688}
print(info.safetensors.total / 1e9)           # 3.0859...

3085938688
{'BF16': 3085938688}
3.085938688


# The Memory Components

There are 5 components:

1. Model Weights
2. Gradients
3. Optimizer States
4. Activations
5. Cuda Overhead

I will explain the calculation for full finetuning. Then we will explore them for LoRA and QLoRA.


---

## Step 1: Model Weights

- What: The actual neural network parameters            

- Size: params × bytes_per_param

- Example: 3B × 2 bytes (FP16) = 6 GB   

Which means that for only weights (we don't calculate the optimization step, activations or cuda yet) we need **6 GB memory**!

In [ ]:
from huggingface_hub import model_info
from transformers import AutoConfig

config = AutoConfig.from_pretrained("Qwen/Qwen2.5-3B")
info = model_info("Qwen/Qwen2.5-3B")


print(f"Hidden Dimension: {config.hidden_size}")
print(f"Number of Layers:  {config.num_hidden_layers}")
print(f"Intermediate Size: {config.intermediate_size}")
print(f"Number of Heads:   {config.num_attention_heads}")
print(f"Vocab Size:        {config.vocab_size}")
print(info.safetensors.total)

Hidden Dimension: 2048
Number of Layers:  36
Intermediate Size: 11008
Number of Heads:   16
Vocab Size:        151936
3085938688


In [ ]:
params = info.safetensors.total
print(f"params: {params}")

params: 3085938688


In [ ]:
weights_gb_32 = params * 4 / 1e9
weights_gb_16 = params * 2 / 1e9

print(f"weights_gb_32: {weights_gb_32}")
print(f"weights_gb_16: {weights_gb_16}")

weights_gb_32: 12.343754752
weights_gb_16: 6.171877376


---

## Step 2: Gradients


What: “How much should each weight change?”. They are created during backpropagation. When they exist:

- ✅ Training
- ❌ Inference

Size: Usually same size as model weights So if:
- Model weights = 6 GB (FP16)
- Gradients = 6 GB (FP16)

Important:
- Gradients are temporary
- Cleared after optimizer.step() (unless accumulated)

In [ ]:
# 2. Gradients (same size as weights, usually FP16)

gradients_gb = params * 2 / 1e9
print(f"gradients_gb: {gradients_gb}")

gradients_gb: 6.171877376


Now, the required GB is: 6+6 = **12GB!** And we haven't calculated the optimizer step yet!

----

## Step 3: Optimizer Step

There are different optimizers let's take a look at SGD and ADAM.

**SGD optimizer:**


    weight = weight - learning_rate × gradient

Gradients are still computed but after optimizer.step() they are used and cleared. No persistent memory.


**Adam optimizer:**

m (momentum) = Exponential moving average of gradients. Smooths out gradient noise, keeps moving in consistent direction

v (variance) = Exponential moving average of squared gradients. Adapts learning rate per parameter (larger gradients → smaller steps)

    m_t = β₁ × m_{t-1} + (1 - β₁) × gradient_t      # Momentum (direction) needs previous m!

    v_t = β₂ × v_{t-1} + (1 - β₂) × gradient_t²       # Variance (adaptive learning rate)  needs previous v!

    m_t* = m_t/(1-β₁^t) # bias correction
    v_t* = v_t/(1-β₂^t) # bias correction

    update = m_t* / (√v_t* + ε)                        # Normalized update

    weight = weight - lr × (update + λ × weight)       # Weight decay

    NOT for AdamW:
    w = w - lr x ( update - lr x  λ × weight)

Must store m and v for EVERY parameter, forever during training

Please look at the formula again.t is the optimization step. In order to calculate m_t (at time t) we need m_t-1. So for each optimization step we need the previous momentum value and we need to keep it.

OPTIMIZER_BYTES_PER_PARAM:


    'adamw':    8,   # m (4) + v (4)
    'adam':     8,   # Same as AdamW
    'sgd':      0,   # No states (just gradient)
    'sgd_mom':  4,   # momentum buffer (4)
    'adam8bit': 2,   # 8-bit quantized states (bitsandbytes)
    'adafactor': 4,  # Factorized second moment (memory efficient)

In [ ]:
optimizer = "adamw"
# 3. Optimizer States
if optimizer == 'adamw':
    # AdamW: 2 states (m, v) × FP32 = 8 bytes per param
    optimizer_gb = params * 8 / 1e9
elif optimizer == 'sgd':
    # SGD with momentum: 1 state × FP32 = 4 bytes per param
    optimizer_gb = params * 4 / 1e9
elif optimizer == 'adam8bit':
    # 8-bit Adam: ~2 bytes per param
    optimizer_gb = params * 2 / 1e9

print(f"optimizer: {optimizer} optimizer_gb: {optimizer_gb}")

optimizer: adamw optimizer_gb: 24.687509504


As you see only the optimizer part needs **24 gb** itself!

Currently total need is: 6+6+24 =**36 GB**. Now we need A100.

----

## Step 4: Activations

It is not activation functions.

Activations are the intermediate outputs of each layer during the forward pass and **everything needed during backpropagation.**

Alternative Terminology:

- Activations
- Saved tensors
- Intermediate cache
- Forward cache
- Activation memory


**Activations in MEMORY context (broad definition)**   

• ALL intermediate tensors saved during forward pass                      
• Includes inputs to every operation, not just activation functions
• "The model uses 4GB of activation memory"                            


**What affects activation size?**

- Batch size: how many training examples you process at the same time in one forward/backward pass.
- Sequence length: how many tokens the model sees per example.
- Hidden size: number of nodes
- Number of layers


      ┌─────────────────────────────────────────────────────────────────────────────┐
      │                    EVERYTHING SAVED FOR BACKWARD PASS                       │
      ├─────────────────────────────────────────────────────────────────────────────┤
      │                                                                             │
      │  In a Transformer, "activations" includes:                                  │
      │                                                                             │
      │  ┌─────────────────────────────────────────────────────────────────────┐    │
      │  │                                                                     │    │
      │  │  1. INPUTS TO LINEAR LAYERS (for weight gradients)                  │    │
      │  │     • Input to Q projection                                         │    │
      │  │     • Input to K projection                                         │    │
      │  │     • Input to V projection                                         │    │
      │  │     • Input to O projection                                         │    │
      │  │     • Input to MLP gate/up projections                              │    │
      │  │     • Input to MLP down projection                                  │    │
      │  │                                                                     │    │
      │  │  2. ATTENTION SCORES (for attention gradient)                       │    │
      │  │     • Q × Kᵀ result                                                 │    │
      │  │     • Softmax output                                                │    │
      │  │                                                                     │    │
      │  │  3. ACTIVATION FUNCTION INPUTS/OUTPUTS                              │    │
      │  │     • Pre-GELU values (or GELU outputs)                             │    │
      │  │     • Pre-softmax values                                            │    │
      │  │                                                                     │    │
      │  │  4. RESIDUAL CONNECTIONS                                            │    │
      │  │     • Values before residual add                                    │    │
      │  │                                                                     │    │
      │  │  5. LAYERNORM INPUTS                                                │    │
      │  │     • Input statistics (mean, variance)                             │    │
      │  │                                                                     │    │
      │  └─────────────────────────────────────────────────────────────────────┘    │
      │                                                                             │
      └─────────────────────────────────────────────────────────────────────────────┘


Why activations are tricky?
- They scale with batch size
- Often larger than model weights

Activation memory ≈
batch_size × seq_len × hidden_size × num_layers × bytes_per_element × activation_multiplier


bytes_per_element = 2 for FP16 / BF16

activation_multiplier ≈ 3–6 (depends on architecture, because per layer you don’t store just one tensor — you store many intermediates.)


Let's calculate the required GB for activations for batch_size = 1 and seq_lentg = 2048. And we will only consider the the hidden dimensions.


In [ ]:
# Only hidden states

batch_size= 1
seq_length = 2048
hidden_dim = config.hidden_size
num_layers = config.num_hidden_layers

activations_gb = (batch_size * seq_length * hidden_dim * num_layers * 2) / 1e9
print(f"activations_gb: {activations_gb}")


activations_gb: 0.301989888


It seems small only 0.3 GB. However:


**If we add attention and MLP intermediates, we will use the multipler**


| Model type         | Multiplier    |
| ------------------ | ------------- |
| Small transformer  | ~3            |
| LLaMA / Qwen style | **4–6**       |
| FlashAttention     | closer to 3–4 |


In [ ]:
activations_gb * 5 # QWEN

1.50994944

It becomes 1.5 GB. Still not so bad.


**Let's scale the batch size:**

In [ ]:
batch = 1
print(f"batch = 1: {activations_gb * 5* batch} GB")
print("")

batch = 2
print(f"batch = 2: {activations_gb * 5* batch} GB")
print("")

batch = 4
print(f"batch = 4: {activations_gb * 5* batch} GB")
print("")

batch = 8
print(f"batch = 8: {activations_gb * 5* batch} GB")
print("")

batch = 16
print(f"batch = 16: {activations_gb * 5* batch} GB")
print("")

batch = 1: 1.50994944 GB

batch = 2: 3.01989888 GB

batch = 4: 6.03979776 GB

batch = 8: 12.07959552 GB

batch = 16: 24.15919104 GB



**👉 This is why batch size kills you first.**

If we use batch_size = 8 then we need 12 GB!


What is current total?

12 GB (weights) + 6 GB (gradient) + 24 GB (Optimizer) + 12 GB (activations) = 54 GB!!!

**Is there a way to decrease this?**

We will use LoRA and see its effect later but before that there is something we can use to decrease the required GB for activation part: it is gradient checkpoint.


**Gradient Checkpoint Effect**

Key idea: Don’t save activations. Recompute them during backward.


| Without checkpointing | With checkpointing |
| --------------------- | ------------------ |
| Store activations     | Store inputs only  |
| High memory           | Low memory         |
| Fast                  | Slower             |


🔥 Activation Memory WITH Gradient Checkpointing

Checkpointing reduces activation memory by roughly: 1 / num_checkpoint_segments



| Setup                       | Activation memory |
| --------------------------- | ----------------- |
| No checkpoint               | 100%              |
| Full checkpoint (per layer) | **~15–25%**       |
| Partial checkpoint          | ~30–50%           |



In [ ]:
# # without Checkpoint:

batch_size= 4
seq_length = 2048
hidden_dim = config.hidden_size
num_layers = config.num_hidden_layers
activation_multiplier = 5


activations_gb = (batch_size * seq_length * hidden_dim * num_layers * 2 * activation_multiplier) / 1e9
print(f"activations_gb: {activations_gb}")


activations_gb: 6.03979776


In [ ]:
# # WITH Checkpoint:
import math

batch_size= 4
seq_length = 2048
hidden_dim = config.hidden_size
num_layers = config.num_hidden_layers
activation_multiplier = 5
num_layers_checkpoint = math.sqrt(num_layers)

activations_gb = (batch_size * seq_length * hidden_dim  * num_layers_checkpoint* 2 * activation_multiplier) / 1e9
print(f"activations_gb: {activations_gb}")


activations_gb: 1.00663296


Remember that activtion multiplier is an assumption. But roughly with gradient checkpoint the activation gb is decresing from 24 to 0.2. Which is a huge win!

**How to enable gradient checkpoint?**

In Hugging Face / PyTorch:

    model.gradient_checkpointing_enable()


Or in Trainer args:

    gradient_checkpointing=True


Gradient checkpointing trades extra computation for much lower GPU memory by recomputing activations instead of storing them.

**Gradient checkpointing = slower training, but much lower memory usage.**

---
### Step 4: Details for Normal vs Gradient Checkpoint

    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                    NORMAL TRAINING (No Checkpointing)                       │
    ├─────────────────────────────────────────────────────────────────────────────┤
    │                                                                             │
    │  FORWARD PASS: Save everything                                              │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Layer 1 ──► SAVE a₁ ──► Layer 2 ──► SAVE a₂ ──► ... ──► Layer 36 ──► Loss │
    │                                                                             │
    │  Memory: [a₁][a₂][a₃][a₄][a₅]...[a₃₆]  ← ALL 36 layers stored!             │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  BACKWARD PASS: Just use saved values                                       │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  ∂L/∂W₃₆ ← use a₃₅ (already saved)                                          │
    │  ∂L/∂W₃₅ ← use a₃₄ (already saved)                                          │
    │  ...                                                                        │
    │  ∂L/∂W₁  ← use x    (already saved)                                         │
    │                                                                             │
    │  ┌─────────────────────────────────────────────────────────────────────┐    │
    │  │  ✓ Fast (no recomputation)                                          │    │
    │  │  ✗ High memory (store all intermediate values)                      │    │
    │  └─────────────────────────────────────────────────────────────────────┘    │
    │                                                                             │
    └─────────────────────────────────────────────────────────────────────────────┘


    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                    GRADIENT CHECKPOINTING                                   │
    ├─────────────────────────────────────────────────────────────────────────────┤
    │                                                                             │
    │  FORWARD PASS: Save only at checkpoints                                     │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Layer 1 ──► SAVE ──► Layer 2 ──► discard ──► ... ──► Layer 6 ──► SAVE     │
    │       ↑                                                    ↑                │
    │   checkpoint 1                                        checkpoint 2          │
    │                                                                             │
    │  Memory: [a₁][  ][  ][  ][  ][a₆][  ][  ][  ][  ][  ][a₁₂]...               │
    │              ↑                    ↑                                         │
    │           discarded            discarded                                    │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  BACKWARD PASS: Recompute when needed!                                      │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Need a₅ for gradient?                                                      │
    │     │                                                                       │
    │     ▼                                                                       │
    │  Don't have it! Must RECOMPUTE:                                             │
    │     │                                                                       │
    │     ▼                                                                       │
    │  Start from checkpoint (a₁) ──► Layer 2 ──► Layer 3 ──► Layer 4 ──► Layer 5│
    │                                                                     ↑       │
    │                                                              Now have a₅!   │
    │     │                                                                       │
    │     ▼                                                                       │
    │  Use a₅ to compute gradient, then discard a₅                                │
    │                                                                             │
    │  ┌─────────────────────────────────────────────────────────────────────┐    │
    │  │  ✗ Slower (must recompute forward passes)                           │    │
    │  │  ✓ Low memory (only store checkpoints)                              │    │
    │  └─────────────────────────────────────────────────────────────────────┘    │
    │                                                                             │
    └─────────────────────────────────────────────────────────────────────────────┘



  



-----

**Step-by-Step Example: 6 Layers, Checkpoint Every 2 Layers**
    
    
    
    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                    DETAILED EXAMPLE                                         │
    ├─────────────────────────────────────────────────────────────────────────────┤
    │                                                                             │
    │  Setup: 6 layers, checkpoint at layers 1, 3, 5                              │
    │                                                                             │
    │  ═══════════════════════════════════════════════════════════════════════    │
    │  FORWARD PASS                                                               │
    │  ═══════════════════════════════════════════════════════════════════════    │
    │                                                                             │
    │  x ──► L1 ──► a₁ ──► L2 ──► a₂ ──► L3 ──► a₃ ──► L4 ──► a₄ ──► L5 ──► a₅ ──► L6 ──► y
    │              SAVE         discard      SAVE         discard      SAVE              │
    │               ✓              ✗          ✓              ✗          ✓               │
    │                                                                             │
    │  Memory after forward: [x][a₁][a₃][a₅][y]  (only 5 values, not 7)           │
    │                                                                             │
    │  ═══════════════════════════════════════════════════════════════════════    │
    │  BACKWARD PASS                                                              │
    │  ═══════════════════════════════════════════════════════════════════════    │
    │                                                                             │
    │  Step 1: Compute ∂L/∂W₆                                                     │
    │  ─────────────────────────                                                  │
    │  Need: a₅ (input to layer 6)                                                │
    │  Have: a₅ ✓ (it's a checkpoint)                                             │
    │  Action: Use a₅ directly                                                    │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Step 2: Compute ∂L/∂W₅                                                     │
    │  ─────────────────────────                                                  │
    │  Need: a₄ (input to layer 5)                                                │
    │  Have: a₄? ✗ (was discarded!)                                               │
    │  Action: RECOMPUTE from checkpoint a₃                                       │
    │                                                                             │
    │          a₃ ──► L4 ──► a₄ (now we have it!)                                 │
    │                                                                             │
    │  Use a₄, then discard it again                                              │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Step 3: Compute ∂L/∂W₄                                                     │
    │  ─────────────────────────                                                  │
    │  Need: a₃ (input to layer 4)                                                │
    │  Have: a₃ ✓ (it's a checkpoint)                                             │
    │  Action: Use a₃ directly                                                    │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Step 4: Compute ∂L/∂W₃                                                     │
    │  ─────────────────────────                                                  │
    │  Need: a₂ (input to layer 3)                                                │
    │  Have: a₂? ✗ (was discarded!)                                               │
    │  Action: RECOMPUTE from checkpoint a₁                                       │
    │                                                                             │
    │          a₁ ──► L2 ──► a₂ (now we have it!)                                 │
    │                                                                             │
    │  Use a₂, then discard it again                                              │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Step 5: Compute ∂L/∂W₂                                                     │
    │  ─────────────────────────                                                  │
    │  Need: a₁ (input to layer 2)                                                │
    │  Have: a₁ ✓ (it's a checkpoint)                                             │
    │  Action: Use a₁ directly                                                    │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Step 6: Compute ∂L/∂W₁                                                     │
    │  ─────────────────────────                                                  │
    │  Need: x (input to layer 1)                                                 │
    │  Have: x ✓ (always saved)                                                   │
    │  Action: Use x directly                                                     │
    │                                                                             │
    └─────────────────────────────────────────────────────────────────────────────┘

---
**Tradeoff**


    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                    THE TRADE-OFF                                            │
    ├─────────────────────────────────────────────────────────────────────────────┤
    │                                                                             │
    │                        MEMORY                         COMPUTE               │
    │                                                                             │
    │  No Checkpointing     ████████████████████████        ████████              │
    │  (save all)           HIGH (all layers)               LOW (1 forward)       │
    │                                                                             │
    │  Checkpointing        ████████                        ████████████████      │
    │  (save some)          LOW (only checkpoints)          HIGH (recompute)      │
    │                                                                             │
    │  ─────────────────────────────────────────────────────────────────────────  │
    │                                                                             │
    │  Quantified for 36-layer model:                                             │
    │                                                                             │
    │  ┌─────────────────────────┬─────────────────┬─────────────────────────┐    │
    │  │  Method                 │  Memory         │  Forward Passes         │    │
    │  ├─────────────────────────┼─────────────────┼─────────────────────────┤    │
    │  │  No checkpointing       │  36 layers      │  1× (normal)            │    │
    │  │  Checkpoint every 6     │  6 layers       │  ~1.5× (50% more)       │    │
    │  │  Checkpoint every 1     │  1 layer        │  ~2× (double)           │    │
    │  │  (extreme)              │  (minimum)      │  (maximum recompute)    │    │
    │  └─────────────────────────┴─────────────────┴─────────────────────────┘    │
    │                                                                             │
    │  ┌─────────────────────────────────────────────────────────────────────┐    │
    │  │                                                                     │    │
    │  │  Optimal checkpoint interval ≈ √(num_layers)                        │    │
    │  │                                                                     │    │
    │  │  For 36 layers: √36 = 6 → checkpoint every 6 layers                 │    │
    │  │  Memory: O(√n) instead of O(n)                                      │    │
    │  │  Compute: ~1.33× instead of 1×                                      │    │
    │  │                                                                     │    │
    │  └─────────────────────────────────────────────────────────────────────┘    │
    │                                                                             │
    └─────────────────────────────────────────────────────────────────────────────┘

---

## Step 5: CUDA Overhead

What: Memory used by CUDA itself and the framework (PyTorch, NCCL, cuDNN).

This includes:
- CUDA context
- Kernel buffers
- Communication buffers (multi-GPU)
- Memory fragmentation

Typical size: ~0.5 GB – 2 GB per GPU

Sometimes more in distributed training

**Important note**

This memory is:
- Not visible in your model code
- Not optional

Often why “it should fit but doesn’t”

In [ ]:
overhead_gb = 0.7
safety_buffer = 0.15


In [ ]:
 # Total
total_gb = weights_gb_16 + gradients_gb + optimizer_gb + activations_gb + overhead_gb
print(f"weights_gb_16: {weights_gb_16}")
print(f"gradients_gb: {gradients_gb}")
print(f"optimizer_gb: {optimizer_gb}")
print(f"activations_gb: {activations_gb}")
print(f"overhead_gb: {overhead_gb}")
print(f"total_gb: {total_gb}")


total_with_buffer_gb = total_gb * (1.0 + safety_buffer)
print(f"total_with_buffer_gb: {total_with_buffer_gb}")


weights_gb_16: 6.171877376
gradients_gb: 6.171877376
optimizer_gb: 24.687509504
activations_gb: 1.00663296
overhead_gb: 0.7
total_gb: 38.737897216
total_with_buffer_gb: 44.548581798399994


---

## Calcuate Full Finetuning

LEt's create a function to see the whole picture for full finetuning.



In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Union

from huggingface_hub import model_info
from transformers import AutoConfig


@dataclass
class MemoryEstimate:
    params: int
    weights_gb: float
    grads_gb: float
    optimizer_gb: float
    activations_gb: float
    overhead_gb: float
    total_gb: float
    total_with_buffer_gb: float
    notes: Dict[str, Any]


def _get_param_count_from_info(info) -> Optional[int]:
    """
    Prefer exact parameter count from HF safetensors metadata if available.
    """
    st = getattr(info, "safetensors", None)
    if st is None:
        return None
    # huggingface_hub SafeTensorsInfo has .total
    total = getattr(st, "total", None)
    if isinstance(total, int) and total > 0:
        return total
    # sometimes it has .parameters dict like {'BF16': int}
    params_dict = getattr(st, "parameters", None)
    if isinstance(params_dict, dict) and params_dict:
        # take first entry
        first = next(iter(params_dict.values()))
        if isinstance(first, int) and first > 0:
            return first
    return None


def _estimate_params_from_config(config) -> int:
    """
    Rough parameter estimate from config. This is approximate.
    Good fallback when safetensors metadata isn't available.
    Assumes: tied embeddings by default, RMSNorm, SwiGLU-style MLP.
    """
    vocab_size = int(getattr(config, "vocab_size"))
    hidden = int(getattr(config, "hidden_size"))
    layers = int(getattr(config, "num_hidden_layers"))
    heads = int(getattr(config, "num_attention_heads"))
    inter = int(getattr(config, "intermediate_size"))
    num_kv_heads = int(getattr(config, "num_key_value_heads", heads))
    tie = bool(getattr(config, "tie_word_embeddings", True))

    head_dim = hidden // heads

    # Embeddings
    emb = vocab_size * hidden

    # Attention weights (GQA-aware)
    q = hidden * hidden
    k = hidden * (num_kv_heads * head_dim)
    v = hidden * (num_kv_heads * head_dim)
    o = hidden * hidden
    attn = q + k + v + o

    # MLP weights (SwiGLU-like: gate, up, down)
    mlp = 3 * hidden * inter

    # Norm scales (approx)
    norms = 2 * hidden

    per_layer = attn + mlp + norms
    transformer = layers * per_layer

    final_norm = hidden
    lm_head = 0 if tie else hidden * vocab_size

    return emb + transformer + final_norm + lm_head


def _bytes_per_param(precision: str) -> int:
    precision = precision.lower()
    if precision in {"fp16", "float16", "bf16", "bfloat16"}:
        return 2
    if precision in {"fp32", "float32"}:
        return 4
    if precision in {"int8", "8bit"}:
        return 1
    raise ValueError(f"Unsupported precision: {precision}")


def _optimizer_bytes_per_param(optimizer: str) -> int:
    """
    Approx bytes/param for optimizer states.
    - adamw: m and v in FP32 => 2 * 4 = 8 bytes/param
    - sgd_momentum: velocity in FP32 => 4 bytes/param
    - adam8bit: rough ~2 bytes/param (varies by implementation)
    """
    opt = optimizer.lower()
    if opt in {"adamw", "adam"}:
        return 8
    if opt in {"sgd", "sgd_momentum", "momentum"}:
        return 4
    if opt in {"adam8bit", "adam_8bit", "8bit_adam"}:
        return 2
    raise ValueError(f"Unsupported optimizer: {optimizer}")


def estimate_full_finetune_gpu_memory(
    model_id: str,
    *,
    batch_size: int = 4,
    seq_length: int = 512,
    precision: str = "bf16",
    optimizer: str = "adamw",
    gradient_checkpointing: bool = True,
    # Activation multipliers (rule-of-thumb knobs)
    activation_multiplier_no_ckpt: float = 5.0,
    activation_multiplier_ckpt: float = 5,
    # Training setup knobs
    gradient_accumulation_steps: int = 1,
    # Overhead knobs
    cuda_overhead_gb: float = 0.7,
    safety_buffer: float = 0.15,
    trust_remote_code: bool = False,
) -> MemoryEstimate:
    """
    Estimate GPU memory for FULL fine-tuning.

    Notes:
    - Uses exact param count from info.safetensors when available.
    - Otherwise falls back to a config-based approximate param estimate.
    - Activation estimate is heuristic (varies with kernels, FlashAttention, etc.).
    - Batch_size here is the *micro-batch per device* (what sits on GPU at once).
      Effective batch = batch_size * gradient_accumulation_steps * num_gpus
    """
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    info = model_info(model_id)

    # --- Params ---
    params_exact = _get_param_count_from_info(info)
    params_source = "info.safetensors.total" if params_exact else "config_estimate"
    params = params_exact if params_exact else _estimate_params_from_config(config)

    # --- Weights / grads / optimizer ---
    w_bytes = _bytes_per_param(precision)
    weights_gb = params * w_bytes / 1e9

    # Gradients typically stored in fp16/bf16 (2 bytes/param) in mixed precision
    grads_gb = params * 2 / 1e9

    opt_bytes = _optimizer_bytes_per_param(optimizer)
    optimizer_gb = params * opt_bytes / 1e9

    # --- Activations ---
    hidden = int(getattr(config, "hidden_size"))
    layers = int(getattr(config, "num_hidden_layers"))

    # micro_batch is what matters for activations
    micro_batch = batch_size

    if gradient_checkpointing:
        import math
        act_mult = activation_multiplier_ckpt
        ckpt_segments = math.sqrt(layers)
        activation_bytes = micro_batch * seq_length * hidden * ckpt_segments * 2 * act_mult

    else:
        # scales with layers
        act_mult = activation_multiplier_no_ckpt
        activation_bytes = micro_batch * seq_length * hidden * layers * 2 * act_mult

    activations_gb = activation_bytes / 1e9

    overhead_gb = float(cuda_overhead_gb)

    total_gb = weights_gb + grads_gb + optimizer_gb + activations_gb + overhead_gb
    total_with_buffer_gb = total_gb * (1.0 + safety_buffer)

    notes = {
        "params_source": params_source,
        "hidden_size": hidden,
        "num_layers": layers,
        "micro_batch_size": micro_batch,
        "seq_length": seq_length,
        "precision": precision,
        "optimizer": optimizer,
        "gradient_checkpointing": gradient_checkpointing,
        "activation_multiplier_used": act_mult,
        "effective_batch_size_per_device": micro_batch * gradient_accumulation_steps,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "warning": (
            "Activation memory is heuristic; FlashAttention/kernels/padding can change it. "
            "Optimizer+grads are usually the dominant terms for full finetune."
        ),
    }

    return MemoryEstimate(
        params=params,
        weights_gb=weights_gb,
        grads_gb=grads_gb,
        optimizer_gb=optimizer_gb,
        activations_gb=activations_gb,
        overhead_gb=overhead_gb,
        total_gb=total_gb,
        total_with_buffer_gb=total_with_buffer_gb,
        notes=notes,
    )


def pretty_print_estimate(est: MemoryEstimate, *, targets_gb=(16, 24, 40, 48, 80)) -> None:
    print("=" * 64)
    print(f"Params: {est.params:,}  ({est.params/1e9:.2f}B)")
    print("-" * 64)
    print(f"Weights:      {est.weights_gb:6.2f} GB")
    print(f"Gradients:    {est.grads_gb:6.2f} GB")
    print(f"Optimizer:    {est.optimizer_gb:6.2f} GB")
    print(f"Activations:  {est.activations_gb:6.2f} GB")
    print(f"CUDA overhead:{est.overhead_gb:6.2f} GB")
    print("-" * 64)
    print(f"TOTAL:        {est.total_gb:6.2f} GB")
    print(f"With buffer:  {est.total_with_buffer_gb:6.2f} GB")
    print("-" * 64)
    for t in targets_gb:
        ok = "✅" if est.total_with_buffer_gb <= t else "❌"
        print(f"Fits in {t:>2} GB? {ok}")
    print("=" * 64)
    print("Notes:", est.notes)


In [ ]:
# Example usage:
est = estimate_full_finetune_gpu_memory(
    "Qwen/Qwen2.5-3B",
    batch_size=4,
    seq_length=2048,
    precision="bf16",
    optimizer="adamw",
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
)

pretty_print_estimate(est)

Params: 3,085,938,688  (3.09B)
----------------------------------------------------------------
Weights:        6.17 GB
Gradients:      6.17 GB
Optimizer:     24.69 GB
Activations:    1.01 GB
CUDA overhead:  0.70 GB
----------------------------------------------------------------
TOTAL:         38.74 GB
With buffer:   44.55 GB
----------------------------------------------------------------
Fits in 16 GB? ❌
Fits in 24 GB? ❌
Fits in 40 GB? ❌
Fits in 48 GB? ✅
Fits in 80 GB? ✅
Notes: {'params_source': 'info.safetensors.total', 'hidden_size': 2048, 'num_layers': 36, 'micro_batch_size': 4, 'seq_length': 2048, 'precision': 'bf16', 'optimizer': 'adamw', 'gradient_checkpointing': True, 'activation_multiplier_used': 5, 'effective_batch_size_per_device': 16, 'gradient_accumulation_steps': 4, 'warning': 'Activation memory is heuristic; FlashAttention/kernels/padding can change it. Optimizer+grads are usually the dominant terms for full finetune.'}


In [ ]:
# pretty_print_estimate(estimate_full_finetune_gpu_memory("meta-llama/Llama-3.1-8B", batch_size=1, seq_length=2048))
pretty_print_estimate(
    estimate_full_finetune_gpu_memory(
        "mistralai/Mistral-7B-v0.1",
        batch_size=1,
        seq_length=4096
        )
    )


Params: 7,241,732,096  (7.24B)
----------------------------------------------------------------
Weights:       14.48 GB
Gradients:     14.48 GB
Optimizer:     57.93 GB
Activations:    0.95 GB
CUDA overhead:  0.70 GB
----------------------------------------------------------------
TOTAL:         88.55 GB
With buffer:  101.83 GB
----------------------------------------------------------------
Fits in 16 GB? ❌
Fits in 24 GB? ❌
Fits in 40 GB? ❌
Fits in 48 GB? ❌
Fits in 80 GB? ❌
Notes: {'params_source': 'info.safetensors.total', 'hidden_size': 4096, 'num_layers': 32, 'micro_batch_size': 1, 'seq_length': 4096, 'precision': 'bf16', 'optimizer': 'adamw', 'gradient_checkpointing': True, 'activation_multiplier_used': 5, 'effective_batch_size_per_device': 1, 'gradient_accumulation_steps': 1, 'warning': 'Activation memory is heuristic; FlashAttention/kernels/padding can change it. Optimizer+grads are usually the dominant terms for full finetune.'}


## Important Note

Activations calculations are used some multipliers which are just estimations. your actual need could be more than that!

In any case, the most important part is the optimization:

    Weights:        6.17 GB
    Gradients:      6.17 GB
    Optimizer:     24.69 GB
    Activations:    1.01 GB
    CUDA overhead:  0.70 GB
    -----------------------
    TOTAL:         38.74 GB
    With buffer:   44.55 GB


Calculator                                  
• Shows: ~40-90 GB needed for 3B-7B models                             
• Conclusion: Need A100-40GB or larger                                 
• Problem: Most people don't have access to these GPUs!

# LoRA Calculation

In [ ]:
import math

# ============================================================================
# LoRA GPU MEMORY CALCULATION
# ============================================================================
from huggingface_hub import model_info

info = model_info("Qwen/Qwen2.5-3B")

# Model info (from HuggingFace)
total_params = info.safetensors.total
print(f"Total base params: {total_params:,}")

# Config values
hidden_size = config.hidden_size
num_layers = config.num_hidden_layers
num_attention_heads = config.num_attention_heads
num_kv_heads = getattr(config, 'num_key_value_heads', num_attention_heads)
intermediate_size = config.intermediate_size
head_dim = hidden_size // num_attention_heads

print(f"hidden_size: {hidden_size}")
print(f"num_layers: {num_layers}")
print(f"num_kv_heads: {num_kv_heads}")
print(f"head_dim: {head_dim}")

Total base params: 3,085,938,688
hidden_size: 2048
num_layers: 36
num_kv_heads: 2
head_dim: 128


=============================================================
## STEP 1: Calculate LoRA Parameters
=============================================================

In [ ]:
lora_rank = 8
target_modules = ['q_proj', 'v_proj']  # Common default

lora_params_per_layer = 0

for module in target_modules:
    print(f"target module is {module}")
    if module == 'q_proj':
        # hidden_size → hidden_size
        in_features = hidden_size
        out_features = hidden_size
    elif module == 'k_proj':
        # hidden_size → kv_dim
        in_features = hidden_size
        out_features = num_kv_heads * head_dim
    elif module == 'v_proj':
        # hidden_size → kv_dim
        in_features = hidden_size
        out_features = num_kv_heads * head_dim
    elif module == 'o_proj':
        # hidden_size → hidden_size
        in_features = hidden_size
        out_features = hidden_size
    elif module in ['gate_proj', 'up_proj']:
        # hidden_size → intermediate_size
        in_features = hidden_size
        out_features = intermediate_size
    elif module == 'down_proj':
        # intermediate_size → hidden_size
        in_features = intermediate_size
        out_features = hidden_size

    print(f"in_features: {in_features}, out_features: {out_features}")

    # LoRA A: (in_features, rank) + LoRA B: (rank, out_features)
    lora_a_params = in_features * lora_rank
    lora_b_params = lora_rank * out_features
    module_lora_params = lora_a_params + lora_b_params

    print(f"  {module}: A({in_features}×{lora_rank}) + B({lora_rank}×{out_features}) = {module_lora_params:,}")
    print("-"*60)
    lora_params_per_layer += module_lora_params

total_lora_params = lora_params_per_layer * num_layers
lora_percentage = (total_lora_params / total_params) * 100

print(f"\nLoRA params per layer: {lora_params_per_layer:,}")
print(f"Total LoRA params: {total_lora_params:,} ({lora_percentage:.3f}% of base)")

target module is q_proj
in_features: 2048, out_features: 2048
  q_proj: A(2048×8) + B(8×2048) = 32,768
------------------------------------------------------------
target module is v_proj
in_features: 2048, out_features: 256
  v_proj: A(2048×8) + B(8×256) = 18,432
------------------------------------------------------------

LoRA params per layer: 51,200
Total LoRA params: 1,843,200 (0.060% of base)


===========================================================
## STEP 2: Base Model Weights (FROZEN, still in memory)
===========================================================


In [ ]:
base_weights_gb = total_params * 2 / 1e9  # BF16/FP16
print(f"\nBase weights (BF16, frozen): {base_weights_gb:.2f} GB")


Base weights (BF16, frozen): 6.17 GB


========================================================
## STEP 3: LoRA Weights (trainable)
=======================================================


In [ ]:

lora_weights_gb = total_lora_params * 2 / 1e9  # BF16/FP16
print(f"LoRA weights (BF16): {lora_weights_gb:.4f} GB")

LoRA weights (BF16): 0.0037 GB


============================================================
## STEP 4: Gradients (LoRA ONLY!)
============================================================

In [ ]:
# Only compute gradients for trainable LoRA parameters!
lora_gradients_gb = total_lora_params * 2 / 1e9  # BF16
print(f"Gradients (LoRA only): {lora_gradients_gb:.4f} GB")

Gradients (LoRA only): 0.0037 GB


===================================================
## STEP 5: Optimizer States (LoRA ONLY!)
===================================================

In [ ]:
optimizer = "adamw"

if optimizer == 'adamw':
    # AdamW: 2 states (m, v) × FP32 = 8 bytes per param
    lora_optimizer_gb = total_lora_params * 8 / 1e9
elif optimizer == 'sgd':
    # SGD with momentum: 1 state × FP32 = 4 bytes per param
    lora_optimizer_gb = total_lora_params * 4 / 1e9
elif optimizer == 'adam8bit':
    # 8-bit Adam: ~2 bytes per param
    lora_optimizer_gb = total_lora_params * 2 / 1e9

print(f"Optimizer ({optimizer}, LoRA only): {lora_optimizer_gb:.4f} GB")

Optimizer (adamw, LoRA only): 0.0147 GB


===================================================
## STEP 6: Activations (similar to full fine-tuning)
===================================================

In [ ]:
batch_size = 4
seq_length = 2048
activation_multiplier = 5

# WITH Checkpoint:
num_layers_checkpoint = math.sqrt(num_layers)
lora_activations_gb = (batch_size * seq_length * hidden_size * num_layers_checkpoint * 2 * activation_multiplier) / 1e9

print(f"Activations (with checkpointing): {lora_activations_gb:.2f} GB")

Activations (with checkpointing): 1.01 GB


================================================================
## STEP 7: CUDA Overhead
===============================================================

In [ ]:
overhead_gb = 0.7
print(f"CUDA overhead: {overhead_gb:.2f} GB")

CUDA overhead: 0.70 GB


===============================================================
## STEP 8: TOTAL
===================================================================


In [ ]:

lora_total_gb = base_weights_gb + lora_weights_gb + lora_gradients_gb + lora_optimizer_gb + lora_activations_gb + overhead_gb
lora_total_with_buffer = lora_total_gb * 1.15  # 15% safety buffer

print("\n" + "="*60)
print("LoRA MEMORY SUMMARY")
print("="*60)
print(f"Base Weights (frozen):  {base_weights_gb:>8.2f} GB")
print(f"LoRA Weights:           {lora_weights_gb:>8.4f} GB")
print(f"Gradients (LoRA only):  {lora_gradients_gb:>8.4f} GB")
print(f"Optimizer (LoRA only):  {lora_optimizer_gb:>8.4f} GB")
print(f"Activations:            {lora_activations_gb:>8.2f} GB")
print(f"CUDA overhead:          {overhead_gb:>8.2f} GB")
print("-"*60)
print(f"TOTAL:                  {lora_total_gb:>8.2f} GB")
print(f"With buffer (15%):      {lora_total_with_buffer:>8.2f} GB")
print("-"*60)
print(f"Fits T4 (16GB)?  {'✅' if lora_total_with_buffer <= 16 else '❌'}")
print(f"Fits L4 (24GB)?  {'✅' if lora_total_with_buffer <= 24 else '❌'}")
print(f"Fits A100 (40GB)? {'✅' if lora_total_with_buffer <= 40 else '❌'}")
print("="*60)


LoRA MEMORY SUMMARY
Base Weights (frozen):      6.17 GB
LoRA Weights:             0.0037 GB
Gradients (LoRA only):    0.0037 GB
Optimizer (LoRA only):    0.0147 GB
Activations:                1.01 GB
CUDA overhead:              0.70 GB
------------------------------------------------------------
TOTAL:                      7.90 GB
With buffer (15%):          9.09 GB
------------------------------------------------------------
Fits T4 (16GB)?  ✅
Fits L4 (24GB)?  ✅
Fits A100 (40GB)? ✅


# Comparison for Full Finetuning & LoRA

In [ ]:
print("\n" + "="*60)
print("COMPARISON: LoRA vs Full Fine-Tuning")
print("="*60)
print(f"                        LoRA          Full FT        Savings")
print(f"Base Weights:       {base_weights_gb:>8.4f} GB      {weights_gb_16:>8.2f} GB    {(1-base_weights_gb/weights_gb_16)*100:>6.2f}%")
print(f"Gradients:          {lora_gradients_gb:>8.4f} GB    {gradients_gb:>8.2f} GB    {(1-lora_gradients_gb/gradients_gb)*100:>6.2f}%")
print(f"Optimizer:          {lora_optimizer_gb:>8.4f} GB    {optimizer_gb:>8.2f} GB    {(1-lora_optimizer_gb/optimizer_gb)*100:>6.2f}%")
print(f"Activations:        {lora_activations_gb:>8.4f} GB    {activations_gb:>8.2f} GB    {(1-lora_activations_gb/activations_gb)*100:>6.2f}%")
print(f"TOTAL:              {lora_total_gb:>8.2f} GB    {total_gb:>8.2f} GB    {(1-lora_total_gb/total_gb)*100:>6.2f}%")
print("="*60)


COMPARISON: LoRA vs Full Fine-Tuning
                        LoRA          Full FT        Savings
Base Weights:         6.1719 GB          6.17 GB      0.00%
Gradients:            0.0037 GB        6.17 GB     99.94%
Optimizer:            0.0147 GB       24.69 GB     99.94%
Activations:          1.0066 GB        1.01 GB      0.00%
TOTAL:                  7.90 GB       38.74 GB     79.61%


# QLoRA Calculation

In [ ]:
import math

# Model info (from HuggingFace)
total_params = info.safetensors.total
print(f"Total base params: {total_params:,}")

# Config values
hidden_size = config.hidden_size
num_layers = config.num_hidden_layers
num_attention_heads = config.num_attention_heads
num_kv_heads = getattr(config, 'num_key_value_heads', num_attention_heads)
intermediate_size = config.intermediate_size
head_dim = hidden_size // num_attention_heads

print(f"hidden_size: {hidden_size}")
print(f"num_layers: {num_layers}")
print(f"num_kv_heads: {num_kv_heads}")
print(f"head_dim: {head_dim}")

Total base params: 3,085,938,688
hidden_size: 2048
num_layers: 36
num_kv_heads: 2
head_dim: 128



=========================================================
## STEP 1: Calculate QLoRA Parameters (same as LoRA)
==========================================================

In [ ]:
lora_rank = 8
target_modules = ['q_proj', 'v_proj']

lora_params_per_layer = 0

for module in target_modules:
    if module == 'q_proj':
        in_features = hidden_size
        out_features = hidden_size
    elif module == 'k_proj':
        in_features = hidden_size
        out_features = num_kv_heads * head_dim
    elif module == 'v_proj':
        in_features = hidden_size
        out_features = num_kv_heads * head_dim
    elif module == 'o_proj':
        in_features = hidden_size
        out_features = hidden_size
    elif module in ['gate_proj', 'up_proj']:
        in_features = hidden_size
        out_features = intermediate_size
    elif module == 'down_proj':
        in_features = intermediate_size
        out_features = hidden_size

    # LoRA A: (in_features, rank) + LoRA B: (rank, out_features)
    lora_a_params = in_features * lora_rank
    lora_b_params = lora_rank * out_features
    module_lora_params = lora_a_params + lora_b_params

    print(f"  {module}: A({in_features}×{lora_rank}) + B({lora_rank}×{out_features}) = {module_lora_params:,}")
    lora_params_per_layer += module_lora_params

total_lora_params = lora_params_per_layer * num_layers
lora_percentage = (total_lora_params / total_params) * 100

print(f"\nLoRA params per layer: {lora_params_per_layer:,}")
print(f"Total LoRA params: {total_lora_params:,} ({lora_percentage:.3f}% of base)")

  q_proj: A(2048×8) + B(8×2048) = 32,768
  v_proj: A(2048×8) + B(8×256) = 18,432

LoRA params per layer: 51,200
Total LoRA params: 1,843,200 (0.060% of base)


============================================================
## STEP 2: Base Model Weights (4-BIT QUANTIZED!) ← KEY DIFFERENCE!
============================================================

In [ ]:
# QLoRA uses 4-bit NormalFloat (NF4) quantization
# 4-bit = 0.5 bytes per parameter
# Plus quantization overhead (scales, zero points) ≈ 15-20%

bytes_per_param_4bit = 0.5
quantization_overhead = 0.15  # 15% overhead for scales/zero points

base_weights_4bit_gb = total_params * bytes_per_param_4bit / 1e9
quantization_overhead_gb = base_weights_4bit_gb * quantization_overhead
base_weights_total_gb = base_weights_4bit_gb + quantization_overhead_gb

print(f"\nBase weights (4-bit NF4): {base_weights_4bit_gb:.2f} GB")
print(f"Quantization overhead (~15%): {quantization_overhead_gb:.2f} GB")
print(f"Base weights total: {base_weights_total_gb:.2f} GB")



Base weights (4-bit NF4): 1.54 GB
Quantization overhead (~15%): 0.23 GB
Base weights total: 1.77 GB
(Compare: FP16 would be 6.17 GB)
(Savings from quantization: 71.2%)


Let's see what we save with quantization:

In [ ]:
# Compare with FP16/BF16
base_weights_fp16_gb = total_params * 2 / 1e9
print(f"(Compare: FP16 would be {base_weights_fp16_gb:.2f} GB)")
print(f"(Savings from quantization: {(1 - base_weights_total_gb/base_weights_fp16_gb)*100:.1f}%)")

(Compare: FP16 would be 6.17 GB)
(Savings from quantization: 71.2%)


=======================================================
## STEP 3: QLoRA Weights (trainable, FP16/BF16 - NOT quantized!)
===========================================================

In [ ]:
# Important: LoRA weights are kept in FP16/BF16 for training!
qlora_weights_gb = total_lora_params * 2 / 1e9
print(f"\nLoRA weights (BF16, trainable): {lora_weights_gb:.4f} GB")


LoRA weights (BF16, trainable): 0.0037 GB


=======================================================
## STEP 4: Gradients (LoRA ONLY!)
===================================================

In [ ]:
qlora_gradients_gb = total_lora_params * 2 / 1e9
print(f"Gradients (LoRA only): {qlora_gradients_gb:.4f} GB")

Gradients (LoRA only): 0.0037 GB



========================================================
## STEP 5: Optimizer States (LoRA ONLY!)
======================================================

In [ ]:
optimizer = "adamw"

if optimizer == 'adamw':
    qlora_optimizer_gb = total_lora_params * 8 / 1e9
elif optimizer == 'sgd':
    qlora_optimizer_gb = total_lora_params * 4 / 1e9
elif optimizer == 'adam8bit':
    qlora_optimizer_gb = total_lora_params * 2 / 1e9

print(f"Optimizer ({optimizer}, LoRA only): {qlora_optimizer_gb:.4f} GB")

Optimizer (adamw, LoRA only): 0.0147 GB


================================================================
## STEP 6: Activations (slightly higher due to dequantization)
===================================================================

In [ ]:
batch_size = 4
seq_length = 2048
activation_multiplier = 5

# WITH Checkpoint:
# QLoRA activations can be slightly higher due to dequantization buffers
# We add ~10-20% to account for this
qlora_activation_overhead = 1.15  # 15% overhead for dequantization

num_layers_checkpoint = math.sqrt(num_layers)
qlora_activations_gb = (batch_size * seq_length * hidden_size * num_layers_checkpoint * 2 * activation_multiplier) / 1e9
qlora_activations_gb = qlora_activations_gb * qlora_activation_overhead  # Add QLoRA overhead

print(f"Activations (with checkpointing + dequant overhead): {qlora_activations_gb:.2f} GB")

Activations (with checkpointing + dequant overhead): 1.16 GB


=============================================================
## STEP 7: CUDA Overhead
===========================================================

In [ ]:
overhead_gb = 0.7
print(f"CUDA overhead: {overhead_gb:.2f} GB")

CUDA overhead: 0.70 GB


===========================================================
## STEP 8: TOTAL
===========================================================

In [ ]:
qlora_total_gb = base_weights_total_gb + qlora_weights_gb + qlora_gradients_gb + qlora_optimizer_gb + qlora_activations_gb + overhead_gb
qlora_total_with_buffer = qlora_total_gb * 1.15  # 15% safety buffer

print("\n" + "="*64)
print("QLoRA MEMORY SUMMARY")
print("="*64)
print(f"Base Weights (4-bit + overhead):  {base_weights_total_gb:>8.2f} GB")
print(f"LoRA Weights (BF16):              {qlora_weights_gb:>8.4f} GB")
print(f"Gradients (LoRA only):            {qlora_gradients_gb:>8.4f} GB")
print(f"Optimizer (LoRA only):            {qlora_optimizer_gb:>8.4f} GB")
print(f"Activations (+ dequant overhead): {qlora_activations_gb:>8.2f} GB")
print(f"CUDA overhead:                    {overhead_gb:>8.2f} GB")
print("-"*64)
print(f"TOTAL:                            {qlora_total_gb:>8.2f} GB")
print(f"With buffer (15%):                {qlora_total_with_buffer:>8.2f} GB")
print("-"*64)
print(f"Fits T4 (16GB)?  {'✅' if qlora_total_with_buffer <= 16 else '❌'}")
print(f"Fits L4 (24GB)?  {'✅' if qlora_total_with_buffer <= 24 else '❌'}")
print(f"Fits A100 (40GB)? {'✅' if qlora_total_with_buffer <= 40 else '❌'}")
print("="*64)



QLoRA MEMORY SUMMARY
Base Weights (4-bit + overhead):      1.77 GB
LoRA Weights (BF16):                0.0037 GB
Gradients (LoRA only):              0.0037 GB
Optimizer (LoRA only):              0.0147 GB
Activations (+ dequant overhead):     1.16 GB
CUDA overhead:                        0.70 GB
----------------------------------------------------------------
TOTAL:                                3.65 GB
With buffer (15%):                    4.20 GB
----------------------------------------------------------------
Fits T4 (16GB)?  ✅
Fits L4 (24GB)?  ✅
Fits A100 (40GB)? ✅


# Comparison for Full Finetuning & LoRA & QLoRA

In [ ]:
print("\n" + "="*64)
print("COMPARISON: QLoRA vs LoRA vs Full Fine-Tuning")
print("="*64)
print(f"                        QLoRA         LoRA        Full FT")
print("-"*64)
print(f"Base Weights:       {base_weights_total_gb:>8.2f} GB   {base_weights_gb:>8.2f} GB   {weights_gb_16:>8.2f} GB")
print(f"LoRA Weights:       {qlora_weights_gb:>8.4f} GB   {lora_weights_gb:>8.4f} GB       -")
print(f"Gradients:          {qlora_gradients_gb:>8.4f} GB   {lora_gradients_gb:>8.4f} GB   {gradients_gb:>8.2f} GB")
print(f"Optimizer:          {qlora_optimizer_gb:>8.4f} GB   {lora_optimizer_gb:>8.4f} GB   {optimizer_gb:>8.2f} GB")
print(f"Activations:        {qlora_activations_gb:>8.2f} GB   {lora_activations_gb:>8.2f} GB   {activations_gb:>8.2f} GB")
print(f"Overhead:           {overhead_gb:>8.2f} GB   {overhead_gb:>8.2f} GB   {overhead_gb:>8.2f} GB")
print("-"*64)
print(f"TOTAL:              {qlora_total_gb:>8.2f} GB   {lora_total_gb:>8.2f} GB   {total_gb:>8.2f} GB")
print("="*64)
print(f"\nMemory Savings:")
print(f"  QLoRA vs Full FT: {(1 - qlora_total_gb/total_gb)*100:.1f}%")
print(f"  QLoRA vs LoRA:    {(1 - qlora_total_gb/lora_total_gb)*100:.1f}%")
print(f"  LoRA vs Full FT:  {(1 - lora_total_gb/total_gb)*100:.1f}%")


COMPARISON: QLoRA vs LoRA vs Full Fine-Tuning
                        QLoRA         LoRA        Full FT
----------------------------------------------------------------
Base Weights:           1.77 GB       6.17 GB       6.17 GB
LoRA Weights:         0.0037 GB     0.0037 GB       -
Gradients:            0.0037 GB     0.0037 GB       6.17 GB
Optimizer:            0.0147 GB     0.0147 GB      24.69 GB
Activations:            1.16 GB       1.01 GB       1.01 GB
Overhead:               0.70 GB       0.70 GB       0.70 GB
----------------------------------------------------------------
TOTAL:                  3.65 GB       7.90 GB      38.74 GB

Memory Savings:
  QLoRA vs Full FT: 90.6%
  QLoRA vs LoRA:    53.7%
  LoRA vs Full FT:  79.6%


# Full Calculation for QLoRA and LoRA

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

from huggingface_hub import model_info
from transformers import AutoConfig


@dataclass
class LoRAMemoryEstimate:
    """Memory estimate for LoRA fine-tuning."""
    base_params: int
    lora_params: int
    lora_percentage: float
    base_weights_gb: float
    lora_weights_gb: float
    gradients_gb: float
    optimizer_gb: float
    activations_gb: float
    overhead_gb: float
    total_gb: float
    total_with_buffer_gb: float
    notes: Dict[str, Any]


@dataclass
class QLoRAMemoryEstimate:
    """Memory estimate for QLoRA fine-tuning."""
    base_params: int
    lora_params: int
    lora_percentage: float
    base_weights_gb: float
    quantization_overhead_gb: float
    lora_weights_gb: float
    gradients_gb: float
    optimizer_gb: float
    activations_gb: float
    overhead_gb: float
    total_gb: float
    total_with_buffer_gb: float
    notes: Dict[str, Any]


def _get_param_count_from_info(info) -> Optional[int]:
    """Get exact parameter count from HuggingFace safetensors metadata."""
    st = getattr(info, "safetensors", None)
    if st is None:
        return None
    total = getattr(st, "total", None)
    if isinstance(total, int) and total > 0:
        return total
    params_dict = getattr(st, "parameters", None)
    if isinstance(params_dict, dict) and params_dict:
        first = next(iter(params_dict.values()))
        if isinstance(first, int) and first > 0:
            return first
    return None


def _estimate_params_from_config(config) -> int:
    """Estimate parameters from config (fallback)."""
    vocab_size = int(getattr(config, "vocab_size"))
    hidden = int(getattr(config, "hidden_size"))
    layers = int(getattr(config, "num_hidden_layers"))
    heads = int(getattr(config, "num_attention_heads"))
    inter = int(getattr(config, "intermediate_size"))
    num_kv_heads = int(getattr(config, "num_key_value_heads", heads))
    tie = bool(getattr(config, "tie_word_embeddings", True))

    head_dim = hidden // heads
    emb = vocab_size * hidden
    q = hidden * hidden
    k = hidden * (num_kv_heads * head_dim)
    v = hidden * (num_kv_heads * head_dim)
    o = hidden * hidden
    attn = q + k + v + o
    mlp = 3 * hidden * inter
    norms = 2 * hidden
    per_layer = attn + mlp + norms
    transformer = layers * per_layer
    final_norm = hidden
    lm_head = 0 if tie else hidden * vocab_size

    return emb + transformer + final_norm + lm_head


def _calculate_lora_params(
    hidden_size: int,
    intermediate_size: int,
    num_layers: int,
    num_attention_heads: int,
    num_kv_heads: int,
    lora_rank: int,
    target_modules: List[str],
) -> int:
    """Calculate total LoRA parameters."""
    head_dim = hidden_size // num_attention_heads
    lora_params_per_layer = 0

    for module in target_modules:
        if module == 'q_proj':
            in_features = hidden_size
            out_features = hidden_size
        elif module == 'k_proj':
            in_features = hidden_size
            out_features = num_kv_heads * head_dim
        elif module == 'v_proj':
            in_features = hidden_size
            out_features = num_kv_heads * head_dim
        elif module == 'o_proj':
            in_features = hidden_size
            out_features = hidden_size
        elif module in ['gate_proj', 'up_proj']:
            in_features = hidden_size
            out_features = intermediate_size
        elif module == 'down_proj':
            in_features = intermediate_size
            out_features = hidden_size
        else:
            continue

        # LoRA A: (in_features, rank) + LoRA B: (rank, out_features)
        lora_params_per_layer += (in_features * lora_rank) + (lora_rank * out_features)

    return lora_params_per_layer * num_layers


def _optimizer_bytes_per_param(optimizer: str) -> int:
    """Get bytes per parameter for optimizer states."""
    opt = optimizer.lower()
    if opt in {"adamw", "adam"}:
        return 8
    if opt in {"sgd", "sgd_momentum", "momentum"}:
        return 4
    if opt in {"adam8bit", "adam_8bit", "8bit_adam"}:
        return 2
    raise ValueError(f"Unsupported optimizer: {optimizer}")


# ============================================================================
# LoRA MEMORY ESTIMATION
# ============================================================================

def estimate_lora_memory(
    model_id: str,
    *,
    batch_size: int = 4,
    seq_length: int = 2048,
    lora_rank: int = 8,
    target_modules: List[str] = ['q_proj', 'v_proj'],
    precision: str = "bf16",
    optimizer: str = "adamw",
    gradient_checkpointing: bool = True,
    activation_multiplier: float = 5.0,
    cuda_overhead_gb: float = 0.7,
    safety_buffer: float = 0.15,
    trust_remote_code: bool = False,
) -> LoRAMemoryEstimate:
    """
    Estimate GPU memory for LoRA fine-tuning.

    Args:
        model_id: HuggingFace model ID (e.g., "Qwen/Qwen2.5-3B")
        batch_size: Micro-batch size per device
        seq_length: Sequence length
        lora_rank: LoRA rank (r)
        target_modules: Which modules to apply LoRA to
        precision: Training precision ("bf16", "fp16", "fp32")
        optimizer: Optimizer type ("adamw", "sgd", "adam8bit")
        gradient_checkpointing: Whether to use gradient checkpointing
        activation_multiplier: Multiplier for activation memory estimate
        cuda_overhead_gb: CUDA context overhead
        safety_buffer: Safety buffer percentage (0.15 = 15%)
        trust_remote_code: Trust remote code for config loading

    Returns:
        LoRAMemoryEstimate with detailed breakdown
    """
    # Load config and info
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    info = model_info(model_id)

    # Get base parameters
    base_params = _get_param_count_from_info(info)
    params_source = "info.safetensors.total" if base_params else "config_estimate"
    if not base_params:
        base_params = _estimate_params_from_config(config)

    # Get config values
    hidden_size = int(getattr(config, "hidden_size"))
    num_layers = int(getattr(config, "num_hidden_layers"))
    intermediate_size = int(getattr(config, "intermediate_size"))
    num_attention_heads = int(getattr(config, "num_attention_heads"))
    num_kv_heads = int(getattr(config, "num_key_value_heads", num_attention_heads))

    # Calculate LoRA parameters
    lora_params = _calculate_lora_params(
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        num_layers=num_layers,
        num_attention_heads=num_attention_heads,
        num_kv_heads=num_kv_heads,
        lora_rank=lora_rank,
        target_modules=target_modules,
    )
    lora_percentage = (lora_params / base_params) * 100

    # Bytes per param for precision
    precision_bytes = {"bf16": 2, "fp16": 2, "fp32": 4}.get(precision.lower(), 2)

    # 1. Base model weights (frozen, but still in memory)
    base_weights_gb = base_params * precision_bytes / 1e9

    # 2. LoRA weights (trainable)
    lora_weights_gb = lora_params * precision_bytes / 1e9

    # 3. Gradients (LoRA only!)
    gradients_gb = lora_params * 2 / 1e9  # Gradients typically in FP16/BF16

    # 4. Optimizer states (LoRA only!)
    opt_bytes = _optimizer_bytes_per_param(optimizer)
    optimizer_gb = lora_params * opt_bytes / 1e9

    # 5. Activations
    if gradient_checkpointing:
        ckpt_segments = math.sqrt(num_layers)
        activations_gb = (batch_size * seq_length * hidden_size * ckpt_segments * 2 * activation_multiplier) / 1e9
    else:
        activations_gb = (batch_size * seq_length * hidden_size * num_layers * 2 * activation_multiplier) / 1e9

    # 6. Overhead
    overhead_gb = cuda_overhead_gb

    # Total
    total_gb = base_weights_gb + lora_weights_gb + gradients_gb + optimizer_gb + activations_gb + overhead_gb
    total_with_buffer_gb = total_gb * (1.0 + safety_buffer)

    notes = {
        "params_source": params_source,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "lora_rank": lora_rank,
        "target_modules": target_modules,
        "batch_size": batch_size,
        "seq_length": seq_length,
        "precision": precision,
        "optimizer": optimizer,
        "gradient_checkpointing": gradient_checkpointing,
    }

    return LoRAMemoryEstimate(
        base_params=base_params,
        lora_params=lora_params,
        lora_percentage=lora_percentage,
        base_weights_gb=base_weights_gb,
        lora_weights_gb=lora_weights_gb,
        gradients_gb=gradients_gb,
        optimizer_gb=optimizer_gb,
        activations_gb=activations_gb,
        overhead_gb=overhead_gb,
        total_gb=total_gb,
        total_with_buffer_gb=total_with_buffer_gb,
        notes=notes,
    )


# ============================================================================
# QLoRA MEMORY ESTIMATION
# ============================================================================

def estimate_qlora_memory(
    model_id: str,
    *,
    batch_size: int = 4,
    seq_length: int = 2048,
    lora_rank: int = 8,
    target_modules: List[str] = ['q_proj', 'v_proj'],
    optimizer: str = "adamw",
    gradient_checkpointing: bool = True,
    activation_multiplier: float = 5.0,
    quantization_overhead: float = 0.15,
    dequant_activation_overhead: float = 1.15,
    cuda_overhead_gb: float = 0.7,
    safety_buffer: float = 0.15,
    trust_remote_code: bool = False,
) -> QLoRAMemoryEstimate:
    """
    Estimate GPU memory for QLoRA fine-tuning.

    Args:
        model_id: HuggingFace model ID (e.g., "Qwen/Qwen2.5-3B")
        batch_size: Micro-batch size per device
        seq_length: Sequence length
        lora_rank: LoRA rank (r)
        target_modules: Which modules to apply LoRA to
        optimizer: Optimizer type ("adamw", "sgd", "adam8bit")
        gradient_checkpointing: Whether to use gradient checkpointing
        activation_multiplier: Multiplier for activation memory estimate
        quantization_overhead: Overhead for quantization scales/zero points (~15%)
        dequant_activation_overhead: Extra activation memory for dequantization (~15%)
        cuda_overhead_gb: CUDA context overhead
        safety_buffer: Safety buffer percentage (0.15 = 15%)
        trust_remote_code: Trust remote code for config loading

    Returns:
        QLoRAMemoryEstimate with detailed breakdown
    """
    # Load config and info
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    info = model_info(model_id)

    # Get base parameters
    base_params = _get_param_count_from_info(info)
    params_source = "info.safetensors.total" if base_params else "config_estimate"
    if not base_params:
        base_params = _estimate_params_from_config(config)

    # Get config values
    hidden_size = int(getattr(config, "hidden_size"))
    num_layers = int(getattr(config, "num_hidden_layers"))
    intermediate_size = int(getattr(config, "intermediate_size"))
    num_attention_heads = int(getattr(config, "num_attention_heads"))
    num_kv_heads = int(getattr(config, "num_key_value_heads", num_attention_heads))

    # Calculate LoRA parameters
    lora_params = _calculate_lora_params(
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        num_layers=num_layers,
        num_attention_heads=num_attention_heads,
        num_kv_heads=num_kv_heads,
        lora_rank=lora_rank,
        target_modules=target_modules,
    )
    lora_percentage = (lora_params / base_params) * 100

    # 1. Base model weights (4-bit quantized!)
    bytes_per_param_4bit = 0.5  # 4-bit = 0.5 bytes
    base_weights_4bit_gb = base_params * bytes_per_param_4bit / 1e9
    quant_overhead_gb = base_weights_4bit_gb * quantization_overhead
    base_weights_gb = base_weights_4bit_gb + quant_overhead_gb

    # 2. LoRA weights (trainable, BF16 - NOT quantized!)
    lora_weights_gb = lora_params * 2 / 1e9

    # 3. Gradients (LoRA only!)
    gradients_gb = lora_params * 2 / 1e9

    # 4. Optimizer states (LoRA only!)
    opt_bytes = _optimizer_bytes_per_param(optimizer)
    optimizer_gb = lora_params * opt_bytes / 1e9

    # 5. Activations (with dequantization overhead)
    if gradient_checkpointing:
        ckpt_segments = math.sqrt(num_layers)
        activations_gb = (batch_size * seq_length * hidden_size * ckpt_segments * 2 * activation_multiplier) / 1e9
    else:
        activations_gb = (batch_size * seq_length * hidden_size * num_layers * 2 * activation_multiplier) / 1e9

    activations_gb = activations_gb * dequant_activation_overhead  # Add QLoRA dequant overhead

    # 6. Overhead
    overhead_gb = cuda_overhead_gb

    # Total
    total_gb = base_weights_gb + lora_weights_gb + gradients_gb + optimizer_gb + activations_gb + overhead_gb
    total_with_buffer_gb = total_gb * (1.0 + safety_buffer)

    notes = {
        "params_source": params_source,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "lora_rank": lora_rank,
        "target_modules": target_modules,
        "batch_size": batch_size,
        "seq_length": seq_length,
        "quantization": "4-bit NF4",
        "optimizer": optimizer,
        "gradient_checkpointing": gradient_checkpointing,
        "quantization_overhead": f"{quantization_overhead*100:.0f}%",
        "dequant_activation_overhead": f"{(dequant_activation_overhead-1)*100:.0f}%",
    }

    return QLoRAMemoryEstimate(
        base_params=base_params,
        lora_params=lora_params,
        lora_percentage=lora_percentage,
        base_weights_gb=base_weights_4bit_gb,
        quantization_overhead_gb=quant_overhead_gb,
        lora_weights_gb=lora_weights_gb,
        gradients_gb=gradients_gb,
        optimizer_gb=optimizer_gb,
        activations_gb=activations_gb,
        overhead_gb=overhead_gb,
        total_gb=total_gb,
        total_with_buffer_gb=total_with_buffer_gb,
        notes=notes,
    )


# ============================================================================
# PRETTY PRINT FUNCTIONS
# ============================================================================

def pretty_print_lora_estimate(est: LoRAMemoryEstimate, targets_gb=(16, 24, 40, 80)) -> None:
    """Pretty print LoRA memory estimate."""
    print("=" * 64)
    print("LoRA MEMORY ESTIMATE")
    print("=" * 64)
    print(f"Base Params:  {est.base_params:,} ({est.base_params/1e9:.2f}B)")
    print(f"LoRA Params:  {est.lora_params:,} ({est.lora_percentage:.3f}% of base)")
    print("-" * 64)
    print(f"Base Weights (frozen):    {est.base_weights_gb:>8.2f} GB")
    print(f"LoRA Weights:             {est.lora_weights_gb:>8.4f} GB")
    print(f"Gradients (LoRA only):    {est.gradients_gb:>8.4f} GB")
    print(f"Optimizer (LoRA only):    {est.optimizer_gb:>8.4f} GB")
    print(f"Activations:              {est.activations_gb:>8.2f} GB")
    print(f"CUDA overhead:            {est.overhead_gb:>8.2f} GB")
    print("-" * 64)
    print(f"TOTAL:                    {est.total_gb:>8.2f} GB")
    print(f"With buffer:              {est.total_with_buffer_gb:>8.2f} GB")
    print("-" * 64)
    for t in targets_gb:
        ok = "✅" if est.total_with_buffer_gb <= t else "❌"
        print(f"Fits in {t:>2} GB? {ok}")
    print("=" * 64)


def pretty_print_qlora_estimate(est: QLoRAMemoryEstimate, targets_gb=(16, 24, 40, 80)) -> None:
    """Pretty print QLoRA memory estimate."""
    print("=" * 64)
    print("QLoRA MEMORY ESTIMATE")
    print("=" * 64)
    print(f"Base Params:  {est.base_params:,} ({est.base_params/1e9:.2f}B)")
    print(f"LoRA Params:  {est.lora_params:,} ({est.lora_percentage:.3f}% of base)")
    print("-" * 64)
    print(f"Base Weights (4-bit):     {est.base_weights_gb:>8.2f} GB")
    print(f"Quantization overhead:    {est.quantization_overhead_gb:>8.2f} GB")
    print(f"LoRA Weights (BF16):      {est.lora_weights_gb:>8.4f} GB")
    print(f"Gradients (LoRA only):    {est.gradients_gb:>8.4f} GB")
    print(f"Optimizer (LoRA only):    {est.optimizer_gb:>8.4f} GB")
    print(f"Activations (+ dequant):  {est.activations_gb:>8.2f} GB")
    print(f"CUDA overhead:            {est.overhead_gb:>8.2f} GB")
    print("-" * 64)
    print(f"TOTAL:                    {est.total_gb:>8.2f} GB")
    print(f"With buffer:              {est.total_with_buffer_gb:>8.2f} GB")
    print("-" * 64)
    for t in targets_gb:
        ok = "✅" if est.total_with_buffer_gb <= t else "❌"
        print(f"Fits in {t:>2} GB? {ok}")
    print("=" * 64)

In [ ]:
# LoRA Example
lora_est = estimate_lora_memory(
    "Qwen/Qwen2.5-3B",
    batch_size=4,
    seq_length=2048,
    lora_rank=8,
    target_modules=['q_proj', 'v_proj'],
    gradient_checkpointing=True,
)
pretty_print_lora_estimate(lora_est)

# QLoRA Example
qlora_est = estimate_qlora_memory(
    "Qwen/Qwen2.5-3B",
    batch_size=4,
    seq_length=2048,
    lora_rank=8,
    target_modules=['q_proj', 'v_proj'],
    gradient_checkpointing=True,
)
pretty_print_qlora_estimate(qlora_est)

LoRA MEMORY ESTIMATE
Base Params:  3,085,938,688 (3.09B)
LoRA Params:  1,843,200 (0.060% of base)
----------------------------------------------------------------
Base Weights (frozen):        6.17 GB
LoRA Weights:               0.0037 GB
Gradients (LoRA only):      0.0037 GB
Optimizer (LoRA only):      0.0147 GB
Activations:                  1.01 GB
CUDA overhead:                0.70 GB
----------------------------------------------------------------
TOTAL:                        7.90 GB
With buffer:                  9.09 GB
----------------------------------------------------------------
Fits in 16 GB? ✅
Fits in 24 GB? ✅
Fits in 40 GB? ✅
Fits in 80 GB? ✅
QLoRA MEMORY ESTIMATE
Base Params:  3,085,938,688 (3.09B)
LoRA Params:  1,843,200 (0.060% of base)
----------------------------------------------------------------
Base Weights (4-bit):         1.54 GB
Quantization overhead:        0.23 GB
LoRA Weights (BF16):        0.0037 GB
Gradients (LoRA only):      0.0037 GB
Optimizer (LoRA onl

# Compare FineTuning Methods for a Single Model

In [ ]:
from typing import List, Tuple

def compare_all_methods(
    model_id: str,
    *,
    batch_size: int = 4,
    seq_length: int = 2048,
    lora_rank: int = 8,
    target_modules: List[str] = ['q_proj', 'v_proj'],
    optimizer: str = "adamw",
    gradient_checkpointing: bool = True,
    activation_multiplier: float = 5.0,
    cuda_overhead_gb: float = 0.7,
    safety_buffer: float = 0.15,
    trust_remote_code: bool = False,
    print_results: bool = True,
) -> Tuple[dict, dict, dict]:
    """
    Compare Full Fine-Tuning, LoRA, and QLoRA memory requirements.

    Returns:
        Tuple of (full_ft_dict, lora_dict, qlora_dict)
    """
    import math
    from huggingface_hub import model_info
    from transformers import AutoConfig

    # Load config and info
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    info = model_info(model_id)

    # Get base parameters
    base_params = _get_param_count_from_info(info)
    if not base_params:
        base_params = _estimate_params_from_config(config)

    # Get config values
    hidden_size = int(getattr(config, "hidden_size"))
    num_layers = int(getattr(config, "num_hidden_layers"))
    intermediate_size = int(getattr(config, "intermediate_size"))
    num_attention_heads = int(getattr(config, "num_attention_heads"))
    num_kv_heads = int(getattr(config, "num_key_value_heads", num_attention_heads))

    # Calculate LoRA parameters
    lora_params = _calculate_lora_params(
        hidden_size=hidden_size,
        intermediate_size=intermediate_size,
        num_layers=num_layers,
        num_attention_heads=num_attention_heads,
        num_kv_heads=num_kv_heads,
        lora_rank=lora_rank,
        target_modules=target_modules,
    )
    lora_percentage = (lora_params / base_params) * 100

    # Activations (with checkpointing)
    if gradient_checkpointing:
        ckpt_segments = math.sqrt(num_layers)
        activations_base = (batch_size * seq_length * hidden_size * ckpt_segments * 2 * activation_multiplier) / 1e9
    else:
        activations_base = (batch_size * seq_length * hidden_size * num_layers * 2 * activation_multiplier) / 1e9

    overhead_gb = cuda_overhead_gb
    opt_bytes = _optimizer_bytes_per_param(optimizer)

    # ========================================================================
    # FULL FINE-TUNING
    # ========================================================================
    full_weights_gb = base_params * 2 / 1e9  # BF16
    full_gradients_gb = base_params * 2 / 1e9
    full_optimizer_gb = base_params * opt_bytes / 1e9
    full_activations_gb = activations_base
    full_total_gb = full_weights_gb + full_gradients_gb + full_optimizer_gb + full_activations_gb + overhead_gb
    full_total_buffer_gb = full_total_gb * (1 + safety_buffer)

    # ========================================================================
    # LoRA
    # ========================================================================
    lora_base_weights_gb = base_params * 2 / 1e9  # BF16 (frozen)
    lora_weights_gb = lora_params * 2 / 1e9
    lora_gradients_gb = lora_params * 2 / 1e9
    lora_optimizer_gb = lora_params * opt_bytes / 1e9
    lora_activations_gb = activations_base
    lora_total_gb = lora_base_weights_gb + lora_weights_gb + lora_gradients_gb + lora_optimizer_gb + lora_activations_gb + overhead_gb
    lora_total_buffer_gb = lora_total_gb * (1 + safety_buffer)

    # ========================================================================
    # QLoRA
    # ========================================================================
    qlora_base_weights_gb = base_params * 0.5 / 1e9  # 4-bit
    qlora_quant_overhead_gb = qlora_base_weights_gb * 0.15  # 15% overhead
    qlora_weights_gb = lora_params * 2 / 1e9  # LoRA in BF16
    qlora_gradients_gb = lora_params * 2 / 1e9
    qlora_optimizer_gb = lora_params * opt_bytes / 1e9
    qlora_activations_gb = activations_base * 1.15  # 15% dequant overhead
    qlora_total_gb = qlora_base_weights_gb + qlora_quant_overhead_gb + qlora_weights_gb + qlora_gradients_gb + qlora_optimizer_gb + qlora_activations_gb + overhead_gb
    qlora_total_buffer_gb = qlora_total_gb * (1 + safety_buffer)

    # ========================================================================
    # RESULTS
    # ========================================================================
    full_ft = {
        "base_weights_gb": full_weights_gb,
        "lora_weights_gb": 0,
        "gradients_gb": full_gradients_gb,
        "optimizer_gb": full_optimizer_gb,
        "activations_gb": full_activations_gb,
        "overhead_gb": overhead_gb,
        "total_gb": full_total_gb,
        "total_buffer_gb": full_total_buffer_gb,
    }

    lora = {
        "base_weights_gb": lora_base_weights_gb,
        "lora_weights_gb": lora_weights_gb,
        "gradients_gb": lora_gradients_gb,
        "optimizer_gb": lora_optimizer_gb,
        "activations_gb": lora_activations_gb,
        "overhead_gb": overhead_gb,
        "total_gb": lora_total_gb,
        "total_buffer_gb": lora_total_buffer_gb,
    }

    qlora = {
        "base_weights_gb": qlora_base_weights_gb + qlora_quant_overhead_gb,
        "lora_weights_gb": qlora_weights_gb,
        "gradients_gb": qlora_gradients_gb,
        "optimizer_gb": qlora_optimizer_gb,
        "activations_gb": qlora_activations_gb,
        "overhead_gb": overhead_gb,
        "total_gb": qlora_total_gb,
        "total_buffer_gb": qlora_total_buffer_gb,
    }

    # ========================================================================
    # PRINT RESULTS
    # ========================================================================
    if print_results:
        print("\n" + "=" * 72)
        print(f"MODEL: {model_id}")
        print("activation_multiplier: 5, cuda_overhead_gb: 0.7, safety_buffer: 0.15")
        print(f"Base Params: {base_params:,} ({base_params/1e9:.2f}B)")
        print(f"LoRA Params: {lora_params:,} ({lora_percentage:.3f}% of base)")
        print(f"Config: batch={batch_size}, seq={seq_length}, rank={lora_rank}, ckpt={gradient_checkpointing}")
        print("=" * 72)
        print(f"{'Component':<24} {'QLoRA':>12} {'LoRA':>12} {'Full FT':>12}")
        print("-" * 72)
        print(f"{'Base Weights:':<24} {qlora['base_weights_gb']:>10.2f} GB {lora['base_weights_gb']:>10.2f} GB {full_ft['base_weights_gb']:>10.2f} GB")
        print(f"{'LoRA Weights:':<24} {qlora['lora_weights_gb']:>10.4f} GB {lora['lora_weights_gb']:>10.4f} GB {'-':>12}")
        print(f"{'Gradients:':<24} {qlora['gradients_gb']:>10.4f} GB {lora['gradients_gb']:>10.4f} GB {full_ft['gradients_gb']:>10.2f} GB")
        print(f"{'Optimizer:':<24} {qlora['optimizer_gb']:>10.4f} GB {lora['optimizer_gb']:>10.4f} GB {full_ft['optimizer_gb']:>10.2f} GB")
        print(f"{'Activations:':<24} {qlora['activations_gb']:>10.2f} GB {lora['activations_gb']:>10.2f} GB {full_ft['activations_gb']:>10.2f} GB")
        print(f"{'Overhead:':<24} {qlora['overhead_gb']:>10.2f} GB {lora['overhead_gb']:>10.2f} GB {full_ft['overhead_gb']:>10.2f} GB")
        print("-" * 72)
        print(f"{'TOTAL:':<24} {qlora['total_gb']:>10.2f} GB {lora['total_gb']:>10.2f} GB {full_ft['total_gb']:>10.2f} GB")
        print(f"{'With Buffer (15%):':<24} {qlora['total_buffer_gb']:>10.2f} GB {lora['total_buffer_gb']:>10.2f} GB {full_ft['total_buffer_gb']:>10.2f} GB")
        print("-" * 72)

        # GPU Fit Check
        print(f"{'GPU Fit:':<24} {'QLoRA':>12} {'LoRA':>12} {'Full FT':>12}")
        for gpu_name, gpu_gb in [("T4 (16GB)", 16), ("L4 (24GB)", 24), ("A100-40GB", 40), ("A100-80GB", 80)]:
            qlora_fit = "✅" if qlora['total_buffer_gb'] <= gpu_gb else "❌"
            lora_fit = "✅" if lora['total_buffer_gb'] <= gpu_gb else "❌"
            full_fit = "✅" if full_ft['total_buffer_gb'] <= gpu_gb else "❌"
            print(f"{gpu_name:<24} {qlora_fit:>12} {lora_fit:>12} {full_fit:>12}")

        print("-" * 72)
        print("Memory Savings:")
        print(f"  QLoRA vs Full FT: {(1 - qlora['total_gb']/full_ft['total_gb'])*100:.1f}%")
        print(f"  LoRA vs Full FT:  {(1 - lora['total_gb']/full_ft['total_gb'])*100:.1f}%")
        print(f"  QLoRA vs LoRA:    {(1 - qlora['total_gb']/lora['total_gb'])*100:.1f}%")
        print("=" * 72)

    return full_ft, lora, qlora

In [ ]:
model_id = "Qwen/Qwen2.5-3B"
batch_size = 4
seq_length = 2048
lora_rank = 8
target_modules=['q_proj', 'v_proj']
gradient_checkpointing = True,

full_ft, lora, qlora = compare_all_methods(
                model_id,
                batch_size=batch_size,
                seq_length=seq_length,
                lora_rank=lora_rank,
                target_modules=target_modules,
                gradient_checkpointing=gradient_checkpointing,
                print_results=True,
                trust_remote_code=True,
            )

print("\n" + "=" * 100)
print("SUMMARY TABLE: GPU Memory Requirements (with 15% buffer)")
print("=" * 100)
print(f"{'Model':<35} {'Params':>10} {'QLoRA':>10} {'LoRA':>10} {'Full FT':>10} {'Best GPU for QLoRA':<20}")
print("-" * 100)


qlora_mem = qlora["total_buffer_gb"]
if qlora_mem <= 16:
    best_gpu = "T4 (16GB)"
elif qlora_mem <= 24:
    best_gpu = "L4 (24GB)"
elif qlora_mem <= 40:
    best_gpu = "A100-40GB"
elif qlora_mem <= 80:
    best_gpu = "A100-80GB"
else:
    best_gpu = "Multi-GPU"

# Get params from full_ft base_weights (base_weights_gb * 1e9 / 2)
params_b = full_ft["base_weights_gb"] / 2

print(f"{model_id:<35} {params_b:>8.1f}B {qlora['total_buffer_gb']:>8.1f}GB "
      f"{lora['total_buffer_gb']:>8.1f}GB {full_ft['total_buffer_gb']:>8.1f}GB {best_gpu:<20}")



MODEL: Qwen/Qwen2.5-3B
activation_multiplier: 5, cuda_overhead_gb: 0.7, safety_buffer: 0.15
Base Params: 3,085,938,688 (3.09B)
LoRA Params: 1,843,200 (0.060% of base)
Config: batch=4, seq=2048, rank=8, ckpt=(True,)
Component                       QLoRA         LoRA      Full FT
------------------------------------------------------------------------
Base Weights:                  1.77 GB       6.17 GB       6.17 GB
LoRA Weights:                0.0037 GB     0.0037 GB            -
Gradients:                   0.0037 GB     0.0037 GB       6.17 GB
Optimizer:                   0.0147 GB     0.0147 GB      24.69 GB
Activations:                   1.16 GB       1.01 GB       1.01 GB
Overhead:                      0.70 GB       0.70 GB       0.70 GB
------------------------------------------------------------------------
TOTAL:                         3.65 GB       7.90 GB      38.74 GB
With Buffer (15%):             4.20 GB       9.09 GB      44.55 GB
---------------------------------------

# Run Comparison for Different Models

In [ ]:
# ============================================================================
# POPULAR MODELS TO COMPARE
# ============================================================================

POPULAR_MODELS = [
    {
        "model_id": "Qwen/Qwen2.5-0.5B",
        "description": "Qwen2.5 0.5B - Smallest Qwen model",
        "why_popular": "Lightweight model for edge devices, mobile deployment, and resource-constrained environments. Great for learning and experimentation.",
    },
    {
        "model_id": "Qwen/Qwen2.5-1.5B",
        "description": "Qwen2.5 1.5B - Small but capable",
        "why_popular": "Good balance of size and capability. Fits on consumer GPUs while maintaining reasonable instruction-following ability.",
    },
    {
        "model_id": "Qwen/Qwen2.5-3B",
        "description": "Qwen2.5 3B - Mid-size model",
        "why_popular": "Sweet spot for fine-tuning on single consumer GPUs. Strong performance for its size, popular for custom chatbots.",
    },
    {
        "model_id": "Qwen/Qwen2.5-7B",
        "description": "Qwen2.5 7B - Full capability",
        "why_popular": "Best quality in the Qwen2.5 series that can still run on consumer hardware. Excellent instruction following and reasoning.",
    },
    {
        "model_id": "mistralai/Mistral-7B-v0.1",
        "description": "Mistral 7B - Efficient architecture",
        "why_popular": "Outperforms Llama 2 13B despite being smaller. Uses sliding window attention and GQA for efficiency. Apache 2.0 license.",
    },
    {
        "model_id": "mistralai/Mixtral-8x7B-v0.1",
        "description": "Mixtral 8x7B - Mixture of Experts",
        "why_popular": "MoE architecture: 46.7B params but only 12.9B active per token. Matches GPT-3.5 quality at lower inference cost.",
    },
    {
        "model_id": "microsoft/phi-2",
        "description": "Phi-2 2.7B - Microsoft's small model",
        "why_popular": "Remarkably capable for its size due to high-quality training data. Great for research and educational purposes.",
    },
    {
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "description": "Phi-3 Mini 3.8B - Latest Microsoft",
        "why_popular": "State-of-the-art small model. Matches Llama 3 8B on many benchmarks despite being half the size. Mobile-friendly.",
    },
]

In [ ]:
def run_popular_models_comparison(
    batch_size: int = 4,
    seq_length: int = 2048,
    lora_rank: int = 8,
    target_modules: List[str] = ['q_proj', 'v_proj'],
    gradient_checkpointing: bool = True,
):
    """Run comparison on all popular models."""

    print("\n" + "=" * 80)
    print("POPULAR MODELS GPU MEMORY COMPARISON")
    print("=" * 80)
    print(f"Config: batch_size={batch_size}, seq_length={seq_length}, lora_rank={lora_rank}")
    print(f"Target modules: {target_modules}, gradient_checkpointing={gradient_checkpointing}")
    print("=" * 80)

    results = []

    for model_info in POPULAR_MODELS:
        model_id = model_info["model_id"]
        description = model_info["description"]
        why_popular = model_info["why_popular"]

        print(f"\n{'─' * 80}")
        print(f"📦 {description}")
        print(f"💡 Why Popular: {why_popular}")
        print(f"{'─' * 80}")

        try:
            full_ft, lora, qlora = compare_all_methods(
                model_id,
                batch_size=batch_size,
                seq_length=seq_length,
                lora_rank=lora_rank,
                target_modules=target_modules,
                gradient_checkpointing=gradient_checkpointing,
                print_results=True,
                trust_remote_code=True,
            )

            results.append({
                "model_id": model_id,
                "description": description,
                "full_ft": full_ft,
                "lora": lora,
                "qlora": qlora,
            })

        except Exception as e:
            print(f"❌ Error loading {model_id}: {e}")
            continue

    # ========================================================================
    # SUMMARY TABLE
    # ========================================================================
    print("\n" + "=" * 100)
    print("SUMMARY TABLE: GPU Memory Requirements (with 15% buffer)")
    print("=" * 100)
    print(f"{'Model':<35} {'Params':>10} {'QLoRA':>10} {'LoRA':>10} {'Full FT':>10} {'Best GPU for QLoRA':<20}")
    print("-" * 100)

    for r in results:
        model_name = r["model_id"].split("/")[-1][:32]

        # Determine best GPU for QLoRA
        qlora_mem = r["qlora"]["total_buffer_gb"]
        if qlora_mem <= 16:
            best_gpu = "T4 (16GB)"
        elif qlora_mem <= 24:
            best_gpu = "L4 (24GB)"
        elif qlora_mem <= 40:
            best_gpu = "A100-40GB"
        elif qlora_mem <= 80:
            best_gpu = "A100-80GB"
        else:
            best_gpu = "Multi-GPU"

        # Get params from full_ft base_weights (base_weights_gb * 1e9 / 2)
        params_b = r["full_ft"]["base_weights_gb"] / 2

        print(f"{model_name:<35} {params_b:>8.1f}B {r['qlora']['total_buffer_gb']:>8.1f}GB "
              f"{r['lora']['total_buffer_gb']:>8.1f}GB {r['full_ft']['total_buffer_gb']:>8.1f}GB {best_gpu:<20}")

    print("=" * 100)

    # ========================================================================
    # GPU RECOMMENDATIONS
    # ========================================================================
    print("\n" + "=" * 80)
    print("GPU RECOMMENDATIONS BY MODEL")
    print("=" * 80)

    gpu_tiers = [
        ("T4 (16GB) - Free Colab/Kaggle", 16),
        ("L4 (24GB) - Cloud budget option", 24),
        ("A100-40GB - Professional", 40),
        ("A100-80GB - Enterprise", 80),
    ]

    for gpu_name, gpu_mem in gpu_tiers:
        print(f"\n🖥️  {gpu_name}:")
        print("-" * 60)

        qlora_fits = [r for r in results if r["qlora"]["total_buffer_gb"] <= gpu_mem]
        lora_fits = [r for r in results if r["lora"]["total_buffer_gb"] <= gpu_mem]
        full_fits = [r for r in results if r["full_ft"]["total_buffer_gb"] <= gpu_mem]

        print(f"  QLoRA: {', '.join([r['model_id'].split('/')[-1] for r in qlora_fits]) or 'None'}")
        print(f"  LoRA:  {', '.join([r['model_id'].split('/')[-1] for r in lora_fits]) or 'None'}")
        print(f"  Full:  {', '.join([r['model_id'].split('/')[-1] for r in full_fits]) or 'None'}")

    print("\n" + "=" * 80)

    return results


# ============================================================================
# RUN THE COMPARISON
# ============================================================================

if __name__ == "__main__":
    results = run_popular_models_comparison(
        batch_size=4,
        seq_length=2048,
        lora_rank=8,
        target_modules=['q_proj', 'v_proj'],
        gradient_checkpointing=True,
    )


POPULAR MODELS GPU MEMORY COMPARISON
Config: batch_size=4, seq_length=2048, lora_rank=8
Target modules: ['q_proj', 'v_proj'], gradient_checkpointing=True

────────────────────────────────────────────────────────────────────────────────
📦 Qwen2.5 0.5B - Smallest Qwen model
💡 Why Popular: Lightweight model for edge devices, mobile deployment, and resource-constrained environments. Great for learning and experimentation.
────────────────────────────────────────────────────────────────────────────────

MODEL: Qwen/Qwen2.5-0.5B
activation_multiplier: 5, cuda_overhead_gb: 0.7, safety_buffer: 0.15
Base Params: 494,032,768 (0.49B)
LoRA Params: 540,672 (0.109% of base)
Config: batch=4, seq=2048, rank=8, ckpt=True
Component                       QLoRA         LoRA      Full FT
------------------------------------------------------------------------
Base Weights:                  0.28 GB       0.99 GB       0.99 GB
LoRA Weights:                0.0011 GB     0.0011 GB            -
Gradients:     